In [1]:
!pip install torch transformers trl peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.8 MB/s eta 0:00:00


In [2]:
!pip install --upgrade --force-reinstall --no-cache-dir torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.0 MB/s eta 0:00:00


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **1 - Baseline Model**
  Using the Qwen2.5-1.5B-Instruct model as the baseline. Analysis will be done in section 4 - Evaluation to compare against the SFT warmed-up and the GRPO post-trained models.

In [6]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# **2 - SFT Warm-Up training**

In [11]:
# Configuring LoRA for more efficient SFT
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16, # Using a big rank for LoRA to leverage Colab's powerful GPU
    lora_alpha=32, # 2*r
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], #
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [12]:
from datasets import load_dataset

dataset = load_dataset("gsm8k", "main", split="train")

def format_prompt(example):
    # Splitting the answer between CoT reasoning + #### number
    # Converting the data to a format more suitable for agents
    parts = example['answer'].split('####')
    reasoning = parts[0].strip()
    final_answer = parts[1].strip()

    text = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": f"<think>{reasoning}</think>#### {final_answer}"}
        ],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

# Answer is already encoded within the second part of the chat, so drop answer column
dataset = dataset.map(format_prompt, remove_columns=["question", "answer"])

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

In [13]:
# SFT warm-up will use 500 samples, and the rest will be for GRPO
SFT_data = dataset.select(range(500))

In [14]:
# Training with SFTTrainer
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    learning_rate=1e-5,
    bf16=True,
    logging_steps=30,
    warmup_steps=0.05,
    dataset_text_field="text"
)

sft_trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=SFT_data,
)

sft_trainer.train()
_ = model.eval() # this has to be called or the model will be stuck in training mode

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
30,1.470635
60,1.133688
90,0.941979
120,0.833595
150,0.758622
180,0.737026


In [16]:
# save SFT warmed-up model for evaluation
model.save_pretrained("/content/drive/MyDrive/CS272/SFT-lora")
tokenizer.save_pretrained("/content/drive/MyDrive/CS272/SFT-lora")

('/content/drive/MyDrive/CS272/SFT-lora/tokenizer_config.json',
 '/content/drive/MyDrive/CS272/SFT-lora/chat_template.jinja',
 '/content/drive/MyDrive/CS272/SFT-lora/tokenizer.json')

# **3 - GRPO Training**

In [24]:
from peft import PeftModel

# Loading base model
grpo_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto"
)

# Loading LoRA matrices from SFT warm-up
grpo_model = PeftModel.from_pretrained(
    grpo_model,
    "/content/drive/MyDrive/CS272/SFT-lora",
    is_trainable=True
)

# Loading tokenizer from SFT warm-up
grpo_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/CS272/SFT-lora")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [25]:
# Prepping the dataset for GRPO
gsm8k = load_dataset("gsm8k", "main", split="train")
# first 500 were used for SFT warm-up, using the next 1000 for GRPO
GRPO_data = gsm8k.select(range(500, 1500))

def format_for_grpo(example):
  return {
      "prompt": [{"role": "user", "content": example["question"]}],
      "answer": example["answer"]
  }

GRPO_data = GRPO_data.map(format_for_grpo, remove_columns=["question"])

In [5]:
import re

def extract_answer(text):
    # extract final answer in #### format first
    match = re.search(r"####\s*([\d,.-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    # Fallback = last number in text in case model doesn't follow
    numbers = re.findall(r"\b\d+\.?\d*\b", text)
    return numbers[-1] if numbers else None

In [26]:
def compute_rewards(completions, answer, **kwargs):
    # completions = a list of generated responses
    # answer = a list of correct answers from the dataset
    # computing rewards for a group of 4 answers

    rewards = []
    for completion, gold in zip(completions, answer):
        # extract text from completion
        if isinstance(completion, list):
            text = completion[0]["content"]
        else:
            text = completion

        # computes format and accuracy reward of the model's response:
        #   0.0 = wrong format and wrong final answer
        #   0.5 = reasoning is placed between the <think> </think> tags
        #     but wrong final answer
        #   1.0 = wrong format but correct final answer
        #   1.5 = correct formnat and correct final answer
        reward = 0.0

        # the response can be broken down into 4 parts:
        #   - before the <think> tag = wrong format
        #   - within the <think> and </think> tags = reasoning = correct format
        #   - after the </think> tag but before #### = wrong format
        #   - after the #### = final answer = correct format
        pattern = r"<think>(.+?)</think>(.*?)####\s*([\d,.-]+)"
        match = re.search(pattern, text, re.DOTALL)

        # Format reward:
        #   <think> must come before </think> and there must be some
        #   text (reasoning) between them.
        #   if correct format, +0.5 for reward
        if match:
            before = text[:match.start()].strip()
            reasoning = match.group(1).strip()
            between = match.group(2).strip()
            if not before and reasoning and not between:
                reward += 0.5

        # Accuracy reward:
        #   if correct final answer, +1.0 for reward
        #   if the model didn't produce an answer after ####, take the last
        #   number in the response, else None (handled by extract_answer)
        pred = extract_answer(text)
        gold_num = extract_answer(gold)
        if pred and gold_num and pred == gold_num:
            reward += 1.0

        rewards.append(reward)
    return rewards

In [27]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./grpo-qwen-gsm8k",

    beta=0.1,                          # KL penalty coefficient
    epsilon=0.2,                       # clip range

    per_device_train_batch_size=4,    # batch size
    num_generations=8,                 # generating 16 responses per prompt
    generation_batch_size=8,
    max_completion_length=256,
    temperature=0.9,                   # for more diverse generations
    gradient_accumulation_steps=4,

    num_train_epochs=1,
    learning_rate=1e-5,
    bf16=True,
    logging_steps=50,
    remove_unused_columns=False,       # keeps "answer" columns for reward function
)

grpo_trainer = GRPOTrainer(
    model=grpo_model,
    args=grpo_config,
    train_dataset=GRPO_data,
    reward_funcs=compute_rewards,
)

grpo_trainer.train()
_ = grpo_model.eval()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
50,-0.008034
100,0.011766
150,0.007084
200,0.016117
250,0.013923
300,-0.007381
350,0.036060
400,0.019248
450,0.001804
500,0.001025


In [16]:
grpo_trainer.save_model("/content/drive/MyDrive/CS272/GRPO-lora")
grpo_tokenizer.save_pretrained("/content/drive/MyDrive/CS272/GRPO-lora")

('/content/drive/MyDrive/CS272/GRPO-lora/tokenizer_config.json',
 '/content/drive/MyDrive/CS272/GRPO-lora/chat_template.jinja',
 '/content/drive/MyDrive/CS272/GRPO-lora/tokenizer.json')

# **4 - Evaluation**

## **4.1 - Accuracy Comparison**
Comparing the accuracies on the full GSM8K dataset's evaluation split between Base model, SFT model, GRPO post-trained model, and alternative GRPO post-trained model.

In [6]:
from tqdm import tqdm
from torch.utils.data import DataLoader

def evaluate(model, tokenizer, dataset, max_new_tokens=256, batch_size=32):
    # evaluating a batch of 32 samples at a time to leverage the powerful Colab GPU
    # prompts the model and collects accuracy over the eval dataset
    correct = 0
    total = 0

    dataloader = DataLoader(dataset, batch_size=batch_size)

    for batch in tqdm(dataloader):
        # Tokenize batch with padding
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": f"{q}\nSolve step by step and end with #### <number>"}],
                tokenize=False,
                add_generation_prompt=True
            )
            for q in batch["question"]
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens= max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode each item in batch
        for i, output in enumerate(outputs):
            input_len = inputs['input_ids'].shape[1]
            generated = tokenizer.decode(
                output[input_len:],
                skip_special_tokens=True
            )
            pred = extract_answer(generated)
            gold = extract_answer(batch["answer"][i])

            if pred and gold and pred == gold:
                correct += 1
            total += 1

        del inputs, outputs
        torch.cuda.empty_cache()

    return correct / total

In [8]:
from datasets import load_dataset

eval_dataset = load_dataset("gsm8k", "main", split="test")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

### **Baseline Model Analysis**
  Evaluating the baseline Qwen2.5-1.5b-Instruct model's performance on the GSM8K dataset with zero-shot prompting

In [ ]:
from peft import PeftModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Loading base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

# Default tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [29]:
gsm8k_accuracy = evaluate(model, tokenizer, eval_dataset)
print(f"Basline Model's GSM8K Accuracy: {gsm8k_accuracy:.2%}")

100%|██████████| 42/42 [06:58<00:00,  9.95s/it]

Basline Model's GSM8K Accuracy: 44.50%


### **SFT Warmed-up Model Analysis**
  Evaluating the SFT warmed-up model's performance on the GSM8K dataset with zero-shot prompting

In [13]:
# Loading base model
sft_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto"
)

# Loading LoRA matrices from SFT warm-up
sft_model = PeftModel.from_pretrained(
    sft_model,
    "/content/drive/MyDrive/CS272/SFT-lora",
    is_trainable=False
)

# Loading tokenizer from SFT warm-up
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/CS272/SFT-lora")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [15]:
gsm8k_accuracy = evaluate(model, tokenizer, eval_dataset)
print(f"SFT-Model's GSM8K Accuracy: {gsm8k_accuracy:.2%}")

100%|██████████| 42/42 [11:32<00:00, 16.49s/it]

SFT-Model's GSM8K Accuracy: 43.44%


### **GRPO Post-Training Model Analysis**
For GRPO, we tested two sets of parameters, one with 8 answer generations and the other with 16. Since 8 < 16, we decided to use a larger KL penalty and a higher learning rate.

In [11]:
# Loading base model
grpo_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto"
)

# Loading LoRA matrices from GRPO
grpo_model = PeftModel.from_pretrained(
    grpo_model,
    "/content/drive/MyDrive/CS272/GRPO-lora",
    is_trainable=False
)

# Loading tokenizer from GRPO
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/CS272/GRPO-lora")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

#### Base parameters:
- For 8 generations
- beta = 0.1
- lr = 1e-5
- max_completion_length = 256

In [17]:
grpo_accuracy = evaluate(grpo_model, grpo_tokenizer, eval_dataset)
print(f"GRPO Model's GSM8K Accuracy: {grpo_accuracy:.2%}")

100%|██████████| 42/42 [11:52<00:00, 16.97s/it]

GRPO Model's GSM8K Accuracy: 47.54%


#### Alternate parameters:
- For 16 generations
- beta = 0.04
- lr = 5e-6
- max_completion_length = 128

In [16]:
grpo_accuracy = evaluate(grpo_model, grpo_tokenizer, eval_dataset)
print(f"GRPO Model's GSM8K Accuracy: {grpo_accuracy:.2%}")

100%|██████████| 42/42 [11:42<00:00, 16.73s/it]

GRPO Model's GSM8K Accuracy: 41.62%


## **4.2 - Reasoning Difference Analysis between SFT and GRPO**
Showcasing a few cases for the difference in reasoning between the SFT and GRPO (8 generations) models. In all cases, GRPO has more concise reasoning and follows the ```<think> *reasoning* </think>``` format without being explicitly prompted to do so. For case 3, GRPO is showing some reward hacking where it's skipping the entire reasoning step because reasoning and format is only worth 0.5 while getting the correct answer is worth 1.0.

In [14]:
def print_question_and_reasoning(model, test):
  # prompts a model given an evaluation question+answer pair
  # prints if the model answered correctly as well as the
  # model's generated response
  messages = [{"role": "user", "content": f"{test['question']}\nSolve step by step and end with #### <number>"}]
  prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  inputs = tokenizer(
      prompt,
      return_tensors="pt",
      truncation=True,
      max_length=512
  ).to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=512,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id
      )

  generated = tokenizer.decode(
      outputs[0][inputs['input_ids'].shape[1]:],
      skip_special_tokens=True
  )

  pred = extract_answer(generated)
  gold = extract_answer(test["answer"])

  print(f"Pred: {pred} | Gold: {gold} | {'Correct' if pred == gold else 'Wrong'}")
  print(f"Generated: {generated}")
  print()

### Case 1: Both SFT and GRPO are wrong

In [21]:
test = eval_dataset[5]

print(f"Question: {test["question"]}\n\n")
print("SFT Output:")
print_question_and_reasoning(sft_model, test)
print("GRPO Output:")
print_question_and_reasoning(grpo_model, test)

Question: Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar wants to buy 16 glasses. How much does he need to pay for them?


SFT Output:
Pred: 16 | Gold: 64 | Wrong
Generated: To determine how much Kylar needs to pay for 16 glasses, we first need to understand the pricing structure:

- The first glass costs $5.
- Every second glass costs 60% of the original price.

Let's break it down step-by-step:

1. **Calculate the cost of the first glass:**
   \[
   \text{Cost of the first glass} = \$5
   \]

2. **Determine the number of second glasses:**
   Since Kylar is buying 16 glasses in total:
   \[
   \text{Number of second glasses} = 16 - 1 = 15
   \]

3. **Calculate the cost of each second glass:**
   Each second glass costs 60% of the original price:
   \[
   \text{Cost per second glass} = 0.60 \times \$5 = \$3
   \]

4. **Calculate the total cost for all second glasses:**
   There are 15 second gl

### Case 2: SFT is wrong and GRPO is correct

In [20]:
test = eval_dataset[3]

print(f"Question: {test["question"]}\n\n")
print("SFT Output:")
print_question_and_reasoning(sft_model, test)
print("GRPO Output:")
print_question_and_reasoning(grpo_model, test)

Question: James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?


SFT Output:
Pred: 3780 | Gold: 540 | Wrong
Generated: To determine the total number of meters James runs in a week, we can break down the problem into smaller steps:

1. Calculate the distance James runs in one session (one set of three sprints):
   - Each sprint is 60 meters.
   - He runs 3 sprints per session.

   \[
   \text{Distance per session} = 3 \times 60 = 180 \text{ meters}
   \]

2. Determine how many sessions he has in a week:
   - He runs 3 sessions per day.
   - There are 7 days in a week.

   \[
   \text{Number of sessions per week} = 3 \times 7 = 21 \text{ sessions}
   \]

3. Calculate the total distance for all sessions in a week:
   - Multiply the distance per session by the number of sessions per week.

   \[
   \text{Total distance per week} = 21 \times 180 = 3780 \text{ meters}
   \]

Therefore, the total number of meters James runs 

### Case 3: SFT is correct and GRPO is wrong
- This case seems to show a sign of reward hacking where the model is prioritizing answering the question (reward = 1.0 if the answer is correct) over reasoning about the answer (reward = 0.5 if it has the correct format).

In [25]:
test = eval_dataset[9]

print(f"Question: {test["question"]}\n\n")
print("SFT Output:")
print_question_and_reasoning(sft_model, test)
print("GRPO Output:")
print_question_and_reasoning(grpo_model, test)

Question: Eliza's rate per hour for the first 40 hours she works each week is $10. She also receives an overtime pay of 1.2 times her regular hourly rate. If Eliza worked for 45 hours this week, how much are her earnings for this week?


SFT Output:
Pred: 460 | Gold: 460 | Correct
Generated: To calculate Eliza's earnings for the week when she worked 45 hours, we need to break down her work into two parts: regular hours (up to 40) and overtime hours.

1. **Regular Hours Earnings**:
   - Regular hours = 40 hours
   - Regular rate = $10/hour

   So, earnings from regular hours = \( 40 \text{ hours} \times \$10/\text{hour} = \$400 \)

2. **Overtime Hours Earnings**:
   - Overtime hours = Total hours - Regular hours = 45 hours - 40 hours = 5 hours
   - Overtime rate = 1.2 times regular rate = \( 1.2 \times \$10/\text{hour} = \$12/\text{hour} \)

   So, earnings from overtime hours = \( 5 \text{ hours} \times \$12/\text{hour} = \$60 \)

3. **Total Earnings**:
   - Total earnings = Earnings f

### Case 4: Both SFT and GRPO are correct

In [24]:
test = eval_dataset[1]

print(f"Question: {test["question"]}\n\n")
print("SFT Output:")
print_question_and_reasoning(sft_model, test)
print("GRPO Output:")
print_question_and_reasoning(grpo_model, test)

Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?


SFT Output:
Pred: 3 | Gold: 3 | Correct
Generated: To solve this problem, let's break it down step by step:

1. **Determine the amount of blue fiber needed:**
   - The problem states that a robe requires 2 bolts of blue fiber.

2. **Calculate the amount of white fiber needed:**
   - It is given that the robe needs half as much white fiber as blue fiber.
   - Therefore, the amount of white fiber required = \( \frac{1}{2} \times 2 \) bolts = 1 bolt.

3. **Add the amounts together to find the total number of bolts:**
   - Total bolts = Blue fiber + White fiber
   - Total bolts = 2 (blue) + 1 (white) = 3 bolts

So, the total number of bolts needed for the robe is **#### 3**.

GRPO Output:
Pred: 3 | Gold: 3 | Correct
Generated: <think>The robe requires 2 bolts of blue fiber.
It also needs half as much white fiber, which is 1/2 * 2 = <<1/2*2=1>>1 bolt of white fiber.
In total